In [4]:
import torch
import torch.nn as nn
from torchvision import models

def load_feature_extractor(device, pretrained_path):
    swin = models.swin_v2_s(weights=None)

    for param in swin.parameters():
        param.requires_grad = False

    num_features = swin.head.in_features
    swin.head = nn.Sequential(
        nn.Linear(num_features, 128),  
        nn.SELU(),
        nn.Dropout(p=0.1),
        nn.Linear(128, 1),  
        nn.Sigmoid()
    )

    swin = swin.to(device)

    checkpoint = torch.load(pretrained_path, map_location=device)
    state_dict = checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint

    # Carrega os pesos normalmente (mesmo com head presente)
    swin.load_state_dict(state_dict)

    # Substitui o head por Identity após carregar
    swin.head = nn.Identity()

    swin.eval()
    return swin

In [5]:
import numpy as np
from tqdm import tqdm

def extract_features(model, dataloader, device):
    features = []
    with torch.no_grad():
        for images, _ in tqdm(dataloader):
            images = images.to(device)
            #images = imalabelsges.unsqueeze(0)
            
            feats = model(images)
            feats = feats.cpu().numpy()
            features.append(feats)
    return np.concatenate(features, axis=0)

In [6]:

from sklearn.svm import OneClassSVM

def train_ocsvm(X_train_feats):
    oc_svm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.05)  # nu é o % esperado de outliers
    oc_svm.fit(X_train_feats)
    return oc_svm

In [7]:

def test_ocsvm(model, oc_svm, dataloader, device):
    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader):
            images = images.to(device)
            feats = model(images)
            feats = feats.cpu().numpy()

            preds = oc_svm.predict(feats)  # -1 = anomalia, 1 = normal

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds)

    return np.array(y_true), np.array(y_pred)

In [8]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate(y_true, y_pred):
    # Convertendo o formato do One-Class SVM para [0, 1] se necessário
    y_pred = np.where(y_pred == 1, 1, 0)  # 1 = psoríase, 0 = anomalia (não psoríase)
    print(classification_report(y_true, y_pred, target_names=["Não Psoríase", "Psoríase"]))
    print(confusion_matrix(y_true, y_pred))


In [12]:
from torchvision import transforms

transforms = transforms.Compose([
        transforms.RandomRotation(50,fill=1),
        transforms.RandomResizedCrop((224,224)),
        transforms.Resize((224,224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.ToTensor(),  # Converte para tensor
    ])


In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("dataset.csv")

# Índices de cada classe
psoriasis_idx = df[df["labels"] == "psoriasis"].index.tolist()
outra_classe_idx = df[df["labels"] != "psoriasis"].index.tolist()

# Divide os índices de psoríase em treino e teste (ex: 80/20)
psoriasis_train_idx, psoriasis_test_idx = train_test_split(
    psoriasis_idx, test_size=0.2, random_state=42, stratify=df.loc[psoriasis_idx, "labels"]
)

train_idx = psoriasis_train_idx
test_idx = psoriasis_test_idx + outra_classe_idx


In [13]:
from application.dataset import CustomDataset 
from torch.utils.data import DataLoader, Subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# DataLoaders
batch_size = 32

train_loader = DataLoader(Subset(CustomDataset.CustomDataset('dataset.csv', transforms, None), train_idx), batch_size=batch_size, shuffle=True, num_workers=1)

# test_loader: imagens COM e SEM psoríase (para teste)

feature_model = load_feature_extractor(device, pretrained_path='/home/king/Documents/PsoriasisEngineering/cnn-skin-model/model_engineering/runs/ml-model-test-1742432834.2114196/model.pt')

X_train_feats = extract_features(feature_model, train_loader, device)

oc_svm = train_ocsvm(X_train_feats)


TypeError: 'NoneType' object is not subscriptable

In [69]:
test_loader = DataLoader(Subset(CustomDataset.CustomDataset('dataset.csv', transforms, None), test_idx), batch_size=batch_size, shuffle=True, num_workers=1)


y_true, y_pred = test_ocsvm(feature_model, oc_svm, test_loader, device)

evaluate(y_true, y_pred)

              precision    recall  f1-score   support

Não Psoríase       0.25      0.06      0.10       352
    Psoríase       0.75      0.94      0.84      1070

    accuracy                           0.72      1422
   macro avg       0.50      0.50      0.47      1422
weighted avg       0.63      0.72      0.65      1422

[[  22  330]
 [  65 1005]]
